<img src="https://raw.githubusercontent.com/swargo98/adaptiSeq/main/adaptiseq/assets/logo.png" alt="adaptiSeq" width="420">

# adaptiSeq — Colab Quickstart

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/swargo98/adaptiSeq/blob/main/adaptiSeq_Colab_Quickstart.ipynb)
[![PyPI](https://img.shields.io/pypi/v/adaptiseq.svg)](https://pypi.org/project/adaptiseq/)

Install **[adaptiSeq](https://pypi.org/project/adaptiseq/)** and test its basics —
metadata, single & **batch downloads**, and the Python API — on a small public
dataset that downloads in **seconds**.

- 📖 Docs: https://swargo98.github.io/adaptiSeq/
- 💻 Source: https://github.com/swargo98/adaptiSeq

**To run everything:** *Runtime → Run all*. No GPU needed. The notebook installs adaptiSeq **0.1.12**.

## 1. Install

adaptiSeq is a Python package plus a few external command-line tools it shells out
to. In Colab, `wget` is already present — we just add `pigz` (used by
the CLI `-g` path). `sra-tools` and Aspera are **not** needed for this quickstart.

In [ ]:
!pip install -q adaptiseq==0.1.12
!apt-get -qq install -y pigz > /dev/null
print("\n✅ install complete")

## 2. Verify the installation

In [ ]:
!adaptiseq --version

import adaptiseq
print("import OK — adaptiseq", adaptiseq.__version__)
print("public API:", [n for n in ("fetch", "resolve", "get_metadata") if hasattr(adaptiseq, n)])

## 3. Fetch metadata (no download)

`-m` resolves an accession across **GSA / SRA / ENA / DDBJ / GEO** and writes a
metadata table — no sequence data is downloaded. We use **PRJNA916347**, a small
project whose runs are only a few KB each.

In [ ]:
!adaptiseq -i PRJNA916347 -m -o demo -Q

import pandas as pd
meta = pd.read_csv("demo/PRJNA916347.metadata.tsv", sep="\t")
print(f"{len(meta)} runs in PRJNA916347")
meta[["run_accession", "fastq_bytes", "fastq_ftp"]].head()

## 4. Batch download (CLI)

The core feature: resolve many accessions **in parallel** and download them through
an **adaptive worker pool**. We take a few of the smallest runs and fetch them as
gzip FASTQ.

Flags: `-g` gzip FASTQ · `-j 4` up to 4 workers · `-k` skip the md5 check · `-Q` quiet.

By default adaptiSeq checks every file's md5 as it lands. The command-line tool then
asks for `sra-tools` at start-up, which Colab does not have, so this cell passes `-k`.
The next section does the verified download from Python, which needs no `sra-tools`.

In [ ]:
runs = meta["run_accession"].head(3).tolist()
with open("batch.txt", "w") as f:
    f.write("\n".join(runs) + "\n")
print("batch list:", runs)

!adaptiseq -i batch.txt -g -k -j 4 -o demo_batch -Q

import os
print("\ndownloaded:", sorted(f for f in os.listdir("demo_batch") if f.endswith(".gz")))

## 5. Batch download (Python API — with md5 verification)

The importable API returns a `FetchResult` and, unlike the `-k` CLI shortcut above,
**md5-verifies** every file against the archive's checksum as it lands, inside the
worker pool (a corrupt copy is deleted and fetched again) — still no `sra-tools`
needed. A run that cannot be delivered and verified is listed in `result.fail_ids`
(and in `fail.log`), never silently skipped. It runs correctly inside Colab's live
event loop.

In [ ]:
import adaptiseq

result = adaptiseq.fetch("batch.txt", outdir="demo_api", gzip=True, jobs=4, adaptive=True, quiet=True)
print("✅ downloaded & md5-verified:", result.success_ids)
print("   failed:", result.fail_ids, "| any failure?", result.failed)

## 6. Single-accession helpers: `resolve` and `get_metadata`

In [ ]:
run = runs[0]

# Where would this run download from, as gzip FASTQ? (no download performed)
print("resolved URL:", adaptiseq.resolve(run, database="ena", gzip=True))

# Parsed metadata rows for a single run:
rows = adaptiseq.get_metadata(run)
print("columns:", len(rows[0]), "| library_strategy:", rows[0].get("library_strategy"))

## 7. Inspect the results

In [ ]:
!ls -la demo_api
print("\n--- success.log ---")
!cat demo_api/success.log
print("\n--- first FASTQ reads ---")
!zcat demo_api/*.fastq.gz | head -4

## Notes & next steps

- **Swap in your own accessions** — any Project / Study / BioSample / Sample /
  Experiment / Run across **GSA / SRA / ENA / DDBJ / GEO**, single or a file of many
  (mixed databases allowed).
- **External tools:** `wget` ships with Colab; we installed `pigz`. For the CLI's md5
  check (without `-k`) and for SRA→FASTQ conversion (`-q`/`-e`), also install
  `sra-tools`: `!apt-get install -y sra-toolkit` (heavier). **Aspera (`-a`) is not
  available in Colab.**
- **Exit status:** on the command line, `0` means every requested run was delivered
  and verified; `1` means at least one was not (it is named in `fail.log`). In
  Python, check `result.failed`.
- **SRA fallback:** a run that ENA cannot serve is fetched from NCBI SRA by default
  (announced; needs `sra-tools`; SRA copies cannot be checked against ENA's md5).
  `--no-sra-fallback` on the command line refuses it.
- **Adaptive controller:** adaptive concurrency is on by default and uses the
  `topdown` controller. Pick another with `--controller topdown|bottomup|greedy|budget|knee`
  (`controller=` in `fetch`), or turn it off with `--no-adaptive -j 20`: on ENA a
  fixed pool is often faster, and the adaptive default needs no value chosen for
  the list. See the docs for how each controller works.
- 📖 Docs: https://swargo98.github.io/adaptiSeq/ · 🐛 Issues:
  https://github.com/swargo98/adaptiSeq/issues